# 第4章　收益率计量

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch04_yield_measures.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch04_yield_measures.ipynb)

复现例4.1（YTM/当期收益率）、例4.2（远期利率）、例4.3（再投资风险），牛顿 vs 二分法对比，即期/远期曲线与 QuantLib 对拍。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi.cashflow import make_cashflows
from fi.pricing import price_bond, ytm, current_yield, forward_rate
from fi import data, plotting
plotting.use_chinese_style()


## 例4.1　从价格反求 YTM


In [ ]:
cfs, ts = make_cashflows(0.03, 3, freq=1, face=100)
price = 97.2249
print(f'到期收益率 YTM   = {ytm(price, cfs, ts, freq=1)*100:.4f}%')
print(f'当期收益率        = {current_yield(3, price)*100:.4f}%')
print(f'票面收益率        = 3.0000%   ->  票面 < 当期 < YTM（折价债）')


## 例4.2　由即期利率求远期利率


In [ ]:
spot = {1: 0.02, 2: 0.025}
f = forward_rate(lambda t: spot[t], 1, 2, freq=1)
print(f'1y1y 远期利率 = {f*100:.4f}%')


## 例4.3　再投资风险：实现收益偏离 YTM


In [ ]:
for r in (0.02, 0.03, 0.04):
    fv = sum(3 * (1 + r) ** (3 - t) for t in (1, 2, 3)) + 100
    rcy = (fv / 100) ** (1 / 3) - 1
    tag = '（= YTM）' if abs(r - 0.03) < 1e-9 else ''
    print(f'再投资@{r:.0%}: 期末财富={fv:8.4f}  实现复合收益={rcy*100:.4f}% {tag}')


### 实现收益率关于再投资利率的曲线（编程实验 7）


In [ ]:
rs = np.linspace(0.00, 0.06, 121)
rcy = [((sum(3*(1+r)**(3-t) for t in (1,2,3)) + 100) / 100) ** (1/3) - 1 for r in rs]
fig, ax = plotting.new_axes()
ax.plot(rs*100, np.array(rcy)*100)
ax.axhline(3, ls=':', color='gray'); ax.axvline(3, ls=':', color='gray')
ax.scatter([3], [3], color='k', zorder=5, label='再投资@YTM 时实现收益=YTM')
ax.set_xlabel('再投资利率 (%)'); ax.set_ylabel('实现复合收益率 (%)')
ax.set_title('再投资利率决定实现收益'); ax.legend()
fig.tight_layout()


## 牛顿法 vs 二分法（编程实验 6）

对同一只债，从一个很差的初值出发，比较两种求解器。


In [ ]:
def ytm_bisection(price, cfs, ts, freq=1, lo=-0.5, hi=1.0, tol=1e-12):
    it = 0
    while it < 500:
        it += 1
        mid = 0.5 * (lo + hi)
        p = price_bond(cfs, ts, mid, freq)
        if abs(p - price) < tol:
            return mid, it
        if p > price:
            lo = mid
        else:
            hi = mid
    return mid, it

y_newton = ytm(price, cfs, ts, freq=1, guess=0.30)   # 故意用很差的初值 30%
y_bis, n_bis = ytm_bisection(price, cfs, ts)
print(f'牛顿法（差初值）  YTM = {y_newton*100:.6f}%')
print(f'二分法            YTM = {y_bis*100:.6f}%  (迭代 {n_bis} 次)')


## 4.6　QuantLib 对拍：价格 -> 收益率


In [ ]:
import QuantLib as ql
ql.Settings.instance().evaluationDate = ql.Date(15, 6, 2026)
sched = ql.Schedule(ql.Date(15, 6, 2026), ql.Date(15, 6, 2029), ql.Period(ql.Annual),
                    ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted,
                    ql.DateGeneration.Backward, False)
dc = ql.ActualActual(ql.ActualActual.ISDA)
bond = ql.FixedRateBond(0, 100.0, sched, [0.03], dc)
y_ql = bond.bondYield(ql.BondPrice(97.2249, ql.BondPrice.Clean), dc, ql.Compounded, ql.Annual)
print(f'fi YTM       = {ytm(97.2249, cfs, ts, freq=1)*100:.4f}%')
print(f'QuantLib YTM = {y_ql*100:.4f}%')


## 4.7　即期与远期曲线


In [ ]:
curve = data.load_sample('cgb_yield_curve')
zt = dict(zip(curve['tenor'], curve['yield_pct'] / 100))
ten = list(curve['tenor'])
fwd_x, fwd_y = [], []
for a, b in zip(ten[:-1], ten[1:]):
    fwd_x.append(b)
    fwd_y.append(forward_rate(lambda t: zt[t], a, b, freq=1) * 100)

fig, ax = plotting.new_axes()
ax.plot(curve['tenor'], curve['yield_pct'], marker='o', label='即期利率 z(t)（样本近似）')
ax.plot(fwd_x, fwd_y, marker='s', ls='--', label='隐含远期利率 f')
ax.set_xlabel('期限（年）'); ax.set_ylabel('利率 (%)')
ax.set_title('图4-1　即期曲线与隐含远期曲线'); ax.legend()
fig.tight_layout()


---

> 小结：YTM 是债券的内部收益率，但只有票息按 YTM 再投资时实现收益才等于 YTM；
> 远期利率由即期利率无套利推出，上行即期曲线下远期高于即期。下一章用 Bootstrap 精确构建即期曲线。
